# Module 05: Linguistic Annotations


# 5.1 Part-of-Speech (POS) Tagging


## 🏷️ POS Tagging Overview

Part-of-speech tagging is the process of assigning grammatical categories (like noun, verb, adjective) to words. 

spaCy provides two types of tags:
1. **`pos_` (Coarse-grained)**: The simple Universal Part-of-Speech tag (e.g., `VERB`, `NOUN`, `ADJ`). These are the same across all languages.
2. **`tag_` (Fine-grained)**: Detailed, language-specific tags (e.g., `VBD` for past tense verb, `NNS` for plural noun).


In [1]:
import spacy
import pandas as pd

nlp = spacy.load("en_core_web_sm")
text = "She rapidly drove the fast red cars."
doc = nlp(text)

data = []
for token in doc:
    data.append({
        "Token": token.text,
        "POS (Coarse)": token.pos_,
        "Tag (Fine)": token.tag_,
        "POS Explanation": spacy.explain(token.pos_),
        "Tag Explanation": spacy.explain(token.tag_)
    })

pd.DataFrame(data)


,Token,POS (Coarse),Tag (Fine),POS Explanation,Tag Explanation
0,She,PRON,PRP,pronoun,"pronoun, personal"
1,rapidly,ADV,RB,adverb,adverb
2,drove,VERB,VBD,verb,"verb, past tense"
3,the,DET,DT,determiner,determiner
4,fast,ADJ,JJ,adjective,"adjective (English), other noun-modifier (Chin..."
5,red,ADJ,JJ,adjective,"adjective (English), other noun-modifier (Chin..."
6,cars,NOUN,NNS,noun,"noun, plural"
7,.,PUNCT,.,punctuation,"punctuation mark, sentence closer"


## 🧠 Context Matters

The Tagger uses statistical context to figure out the correct POS tag. Words can have different tags depending on how they are used.


In [2]:
text1 = "I will book the flight."
text2 = "I read a great book."

print(f"In '{text1}', 'book' is a: {nlp(text1)[2].pos_}")
print(f"In '{text2}', 'book' is a: {nlp(text2)[4].pos_}")


In 'I will book the flight.', 'book' is a: VERB
In 'I read a great book.', 'book' is a: NOUN



<br><br>

---

<br><br>


# 5.2 Lemmatization


## 🪓 What is Lemmatization?

Lemmatization is the process of reducing a word to its base or dictionary form (the "lemma").
- `am`, `are`, `is` ➡️ `be`
- `mice` ➡️ `mouse`
- `running` ➡️ `run`

Lemmatization is generally much better than **stemming** (which just chops off the ends of words, often creating non-words like `runn`), because lemmatization understands the actual vocabulary and grammar rules.


In [1]:
import spacy
import pandas as pd

nlp = spacy.load("en_core_web_sm")
text = "The striped bats are hanging on their feet for best."
doc = nlp(text)

data = []
for token in doc:
    data.append({
        "Token": token.text,
        "Lemma": token.lemma_,
        "POS": token.pos_
    })

pd.DataFrame(data)


,Token,Lemma,POS
0,The,the,DET
1,striped,striped,ADJ
2,bats,bat,NOUN
3,are,be,AUX
4,hanging,hang,VERB
5,on,on,ADP
6,their,their,PRON
7,feet,foot,NOUN
8,for,for,ADP
9,best,good,ADJ


## 🛠️ The Lemmatizer Component

In spaCy 3.x, the Lemmatizer is a standalone pipeline component. It usually relies on the POS tags to figure out the correct lemma. For instance, "meeting" could be a verb (`meet`) or a noun (`meeting`).


In [2]:
text1 = "I am meeting my boss." # verb
text2 = "The meeting was long." # noun

print(f"Verb lemma: {nlp(text1)[2].lemma_}")
print(f"Noun lemma: {nlp(text2)[1].lemma_}")


Verb lemma: meet
Noun lemma: meeting



<br><br>

---

<br><br>


# 5.3 Dependency Parsing


## 🌳 Syntactic Dependencies

Dependency parsing extracts the grammar tree of a sentence, showing which words depend on other words.

Every sentence has a **root** (usually the main verb). Every other word is attached to the root, or to another word that eventually attaches to the root.

- `token.head`: The parent word.
- `token.dep_`: The dependency relation (e.g., `nsubj` for nominal subject, `dobj` for direct object).


In [1]:
import spacy
import pandas as pd

nlp = spacy.load("en_core_web_sm")
doc = nlp("The clever dog quickly chased the red ball.")

data = []
for token in doc:
    data.append({
        "Token": token.text,
        "Dependency": token.dep_,
        "Head Token": token.head.text,
        "Explanation": spacy.explain(token.dep_)
    })

pd.DataFrame(data)


,Token,Dependency,Head Token,Explanation
0,The,det,dog,determiner
1,clever,amod,dog,adjectival modifier
2,dog,nsubj,chased,nominal subject
3,quickly,advmod,chased,adverbial modifier
4,chased,ROOT,chased,root
5,the,det,ball,determiner
6,red,compound,ball,compound
7,ball,dobj,chased,direct object
8,.,punct,chased,punctuation


## 🧭 Navigating the Parse Tree

You can easily extract sub-trees to find the context of a specific word using `token.children`, `token.lefts`, `token.rights`, and `token.subtree`.


In [2]:
# Let's find the main verb (the root)
root = [token for token in doc if token.head == token][0]
print(f"Sentence Root: {root.text}\n")

# Let's find the subject and object of the verb
for child in root.children:
    if child.dep_ == "nsubj":
        print(f"Subject: {child.text}")
    elif child.dep_ == "dobj":
        print(f"Direct Object: {child.text}")


Sentence Root: chased

Subject: dog
Direct Object: ball


In [3]:
# We can also extract the full subtree for a word
# For example, the object is 'ball', but the full object phrase is 'the red ball'
obj = [token for token in doc if token.dep_ == "dobj"][0]

# subtree returns a generator, so we convert it to a list, then join
full_phrase = " ".join([t.text for t in obj.subtree])
print(f"\nFull object phrase: {full_phrase}")



Full object phrase: the red ball



<br><br>

---

<br><br>


# 5.4 Morphological Analysis


## 🧬 Morphology

Morphology deals with the internal structure of words (inflection). While the POS tag might tell you a word is a verb, the morphology tells you it's 3rd person singular, present tense.

You access this via `token.morph`, which returns a `MorphAnalysis` object.


In [1]:
import spacy
nlp = spacy.load("en_core_web_sm")

doc = nlp("She reads books.")

for token in doc:
    print(f"{token.text:<10} | {token.morph}")


She        | Case=Nom|Gender=Fem|Number=Sing|Person=3|PronType=Prs
reads      | Number=Sing|Person=3|Tense=Pres|VerbForm=Fin
books      | Number=Plur
.          | PunctType=Peri


## 🔍 Accessing Specific Morphological Features

You can extract specific features (like Tense, Number, Person) using the `.get()` method. It returns a list of strings (since some features can have multiple values).


In [2]:
verb = doc[1] # 'reads'

print(f"Verb: {verb.text}")
print(f"Tense: {verb.morph.get('Tense')}")
print(f"Number: {verb.morph.get('Number')}")
print(f"Person: {verb.morph.get('Person')}")

noun = doc[2] # 'books'
print(f"\nNoun: {noun.text}")
print(f"Number: {noun.morph.get('Number')}")


Verb: reads
Tense: ['Pres']
Number: ['Sing']
Person: ['3']

Noun: books
Number: ['Plur']



<br><br>

---

<br><br>


# 5.5 Noun Chunks


## 📦 Extracting Noun Phrases

Noun chunks are "base noun phrases" – flat phrases that have a noun as their head. Think of noun chunks as a noun plus the words describing the noun (like adjectives and determiners).

Instead of manually navigating the dependency tree to find adjectives connected to nouns, spaCy provides `doc.noun_chunks` which does it automatically!


In [1]:
import spacy
import pandas as pd

nlp = spacy.load("en_core_web_sm")
text = "Autonomous cars shift insurance liability toward autonomous vehicle manufacturers."
doc = nlp(text)

data = []
for chunk in doc.noun_chunks:
    data.append({
        "Chunk": chunk.text,
        "Root Noun": chunk.root.text,
        "Root Dependency": chunk.root.dep_,
        "Root Head": chunk.root.head.text
    })

pd.DataFrame(data)


,Chunk,Root Noun,Root Dependency,Root Head
0,Autonomous cars,cars,nsubj,shift
1,insurance liability,liability,dobj,shift
2,autonomous vehicle manufacturers,manufacturers,pobj,toward


## ⚙️ How it works
- `chunk.text`: The full phrase (e.g., 'Autonomous cars')
- `chunk.root`: The main noun the chunk is built around (e.g., 'cars')
- `chunk.root.dep_`: The role of the noun in the sentence (e.g., 'nsubj' - the subject shifting liability)
- `chunk.root.head`: The verb or word the noun belongs to (e.g., 'shift')

This is incredibly useful for quickly extracting Key Phrases from documents without needing to write complex dependency rules!



<br><br>

---

<br><br>


# 5.6 Visualizing with displaCy


## 🎨 Introduction to displaCy

Looking at tables of data is helpful, but looking at a visual tree is much easier for understanding syntax. spaCy comes with a built-in visualizer called **displaCy**.

displaCy can visualize:
1. Dependency parse trees
2. Named Entities


In [1]:
import spacy
from spacy import displacy

nlp = spacy.load("en_core_web_sm")
doc = nlp("The clever dog quickly chased the red ball.")


## 🌳 Visualizing Dependencies

To view the dependency tree in a Jupyter notebook, use `displacy.render(doc, style='dep', jupyter=True)`.

*(Note: If you run this as a script outside Jupyter, use `displacy.serve()`, which starts a small web server to view the output in your browser).*


In [2]:
# Set 'distance' to adjust the spacing between words
options = {"distance": 110, "compact": False, "color": "#ffffff", "bg": "#09a3d5", "font": "Source Sans Pro"}

displacy.render(doc, style="dep", jupyter=True, options=options)


## 🏷️ Visualizing Named Entities

displaCy can also highlight Named Entities directly in the text. This is fantastic for reviewing the accuracy of your NER models.


In [3]:
doc2 = nlp("Apple is looking at buying U.K. startup for $1 billion in 2024.")

displacy.render(doc2, style="ent", jupyter=True)


### Customizing Entity Colors
You can customize the colors of specific entities by passing an options dictionary.


In [4]:
colors = {"ORG": "linear-gradient(90deg, #aa9cfc, #fc9ce7)", "MONEY": "lightgreen"}
options = {"ents": ["ORG", "MONEY"], "colors": colors}

# By setting 'ents', we tell displaCy to ONLY highlight ORG and MONEY, ignoring GPE and DATE
displacy.render(doc2, style="ent", jupyter=True, options=options)


## 🎉 Summary of Module 5

You've now explored the deep linguistic capabilities of spaCy!
- You understand coarse (`pos_`) and fine-grained (`tag_`) part-of-speech tags.
- You know how to extract `lemmas` and complex `morphological` data.
- You learned how to navigate the syntactic `dependency tree` and easily extract base `noun_chunks`.
- You made everything beautiful with `displaCy`.

In **Module 6**, we will focus entirely on **Named Entity Recognition (NER)**.
